# 03 — Create an ACORN edge classifier

**Student notebook · 45–60 minutes · CPU only**

Build `TinyEdgeMLP`, test its tensor contract, register it in a disposable copy of the pinned ACORN source, and select it through real YAML.

## How to use this notebook

Run cells in order with **Shift+Enter**. The disposable source copy exists only for this kernel session, so the pinned submodule remains clean. Fill the six exercise cells; the final cells run the registered model through ACORN.

In [ ]:
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys
import tempfile

import torch
import yaml

ROOT = Path.cwd()
if not (ROOT / "vendor" / "acorn").is_dir():
    ROOT = ROOT.parent.resolve()
DATA = ROOT / "tutorial_data" / "edge_classifier"
event = torch.load(sorted((DATA / "trainset").glob("*.pyg"))[0], map_location="cpu", weights_only=False)
node_features = ["hit_r", "hit_phi", "hit_z"]
print(event)

## 1. One raw logit per candidate edge

An edge classifier receives `N` nodes with `F` selected features and `E` candidate edges. Each edge has two endpoints, so a plain endpoint MLP receives `2F` values and returns `E` raw logits. The base class applies the sigmoid where probabilities are needed; `forward` must not apply it.

In [ ]:
input_width = None  # TODO: two endpoints times the number of features
assert input_width == 6

## 2. Stack named node features

ACORN stores features as separate node-like tensors and lists their names in `hparams`. Stacking them produces the `[N, F]` matrix consumed by a model.

In [ ]:
x = None  # TODO: stack event[name] for each configured feature
print("nodes:", x.shape)
assert x.shape == (8, 3)
assert torch.isfinite(x).all()

## 3. Gather both edge endpoints

The two rows of `edge_index` provide source and destination node indices. Indexing `x` with them produces two `[E, F]` tensors.

In [ ]:
start, end = None, None       # TODO: unpack event.edge_index
source_features = None          # TODO: gather x at start
destination_features = None     # TODO: gather x at end
assert source_features.shape == destination_features.shape == (12, 3)
assert torch.equal(source_features[0], x[0])
assert torch.equal(destination_features[0], x[2])

## 4. Convert endpoint features into logits

Concatenate the endpoints to `[E, 2F]`, then use a small MLP ending in one output. Removing only the final singleton dimension gives `[E]`, the contract expected by `EdgeClassifierStage`.

In [ ]:
edge_input = None  # TODO: concatenate endpoint features along the last dimension
network = None     # TODO: Linear(6, 8), ReLU, Linear(8, 1)
logits = None      # TODO: network(edge_input), then squeeze the last dimension
assert edge_input.shape == (12, 6)
assert logits.shape == (12,)
assert torch.isfinite(logits).all()

## 5. Put the computation in an ACORN subclass

Layers belong in `__init__`; tensor computation belongs in `forward`. Calling `super().__init__(hparams)` activates the base stage's data, loss, optimizer, validation, checkpoint, and inference behavior. The module text below is also what we will place in ACORN's model package.

In [ ]:
MODEL_SOURCE = r'''import torch
from ..edge_classifier_stage import EdgeClassifierStage


class TinyEdgeMLP(EdgeClassifierStage):
    def __init__(self, hparams):
        super().__init__(hparams)
        n_features = None  # TODO: number of configured node features
        self.network = None  # TODO: Linear(2*n_features, hidden), ReLU, Linear(hidden, 1)

    def forward(self, batch):
        x = None  # TODO: stack configured batch features along the last dimension
        start, end = None, None  # TODO: unpack batch.edge_index
        edge_input = None  # TODO: concatenate source and destination rows of x
        return None  # TODO: network output with its final size-1 dimension removed
'''

compile(MODEL_SOURCE, "tiny_edge_mlp.py", "exec")
assert "class TinyEdgeMLP(EdgeClassifierStage)" in MODEL_SOURCE
assert "torch.sigmoid" not in MODEL_SOURCE

## 6. Register the class for YAML lookup

ACORN resolves `stage: edge_classifier` and `model: TinyEdgeMLP` with `str_to_class`. Therefore the class file must live in the stage package and the stage's `__init__.py` must import and export it. We perform those edits in a temporary copy, never in the pinned submodule.

In [ ]:
scratch = tempfile.TemporaryDirectory(prefix="acorn_tutorial_03_")
SCRATCH = Path(scratch.name)
ACORN_COPY = SCRATCH / "acorn-source"
shutil.copytree(ROOT / "vendor" / "acorn", ACORN_COPY)
model_path = ACORN_COPY / "acorn" / "stages" / "edge_classifier" / "models" / "tiny_edge_mlp.py"
model_path.write_text(MODEL_SOURCE)
init_path = ACORN_COPY / "acorn" / "stages" / "edge_classifier" / "__init__.py"

In [ ]:
registration = None  # TODO: import TinyEdgeMLP and append its name to __all__
assert "TinyEdgeMLP" in registration
init_path.write_text(init_path.read_text() + registration)

scratch_env = os.environ.copy()
scratch_env["PYTHONPATH"] = str(ACORN_COPY) + os.pathsep + scratch_env.get("PYTHONPATH", "")
check = subprocess.run(
    [sys.executable, "-c", "from acorn.core.core_utils import str_to_class; print(str_to_class('edge_classifier', 'TinyEdgeMLP').__name__)"],
    env=scratch_env, cwd=ROOT, text=True, capture_output=True, timeout=60, check=True,
)
print(check.stdout.strip())
assert "TinyEdgeMLP" in check.stdout

## 7. Select the new model from YAML

Nothing in ACORN's CLI needs a special case. Changing `model` to the exported class name selects it; the inherited stage still supplies training and checkpointing. This smoke run overfits two toy events and runs inference from the resulting checkpoint.

In [ ]:
stage_dir = SCRATCH / "tiny_edge_mlp_run"
train_config = yaml.safe_load((ROOT / "configs" / "02_interaction_gnn_train.yaml").read_text())
train_config.update({
    "model": "TinyEdgeMLP", "input_dir": str(DATA), "stage_dir": str(stage_dir),
    "data_split": [2, 1, 0], "hidden": 8, "max_epochs": 15,
})
train_path = SCRATCH / "tiny_train.yaml"
train_path.write_text(yaml.safe_dump(train_config, sort_keys=False))

def run_scratch(operation, path):
    result = subprocess.run(
        [sys.executable, "-m", "acorn.core.entrypoint_stage", operation, str(path)],
        env=scratch_env, cwd=ROOT, text=True, capture_output=True, timeout=240, check=False,
    )
    print("$ acorn", operation, path.name)
    print("\n".join((result.stdout + result.stderr).splitlines()[-12:]))
    result.check_returncode()

run_scratch("train", train_path)
assert list((stage_dir / "artifacts").glob("*.ckpt"))

infer_config = {key: train_config[key] for key in [
    "stage", "model", "input_dir", "stage_dir", "project", "accelerator",
    "devices", "nodes", "data_split", "variable_with_prefix", "num_workers",
]}
infer_path = SCRATCH / "tiny_infer.yaml"
infer_path.write_text(yaml.safe_dump(infer_config, sort_keys=False))
run_scratch("infer", infer_path)
output_path = sorted((stage_dir / "trainset").glob("*.pyg"))[0]
output_event = torch.load(output_path, map_location="cpu", weights_only=False)
assert output_event.edge_scores.shape == output_event.edge_y.shape
assert torch.isfinite(output_event.edge_scores).all()

## 8. Recap and contribution checklist

A new ACORN edge classifier inherits `EdgeClassifierStage`, calls `super()`, creates layers in `__init__`, returns one raw finite logit per edge from `forward`, is imported by the stage package, is named under `model:` in YAML, and is smoke-tested on a tiny dataset before expensive training.